In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
DATA = Path("data")

## The data: MNIST 0s and 1s, shrunk to 6 x 6
Target y = 1 for a "1", y = 0 for a "0". Shrinking keeps the shapes as small as in the text.

In [2]:
(x_train, y_train), _ = keras.datasets.mnist.load_data()
keep = np.isin(y_train, [0, 1])
small = np.array([np.array(Image.fromarray(im).resize((6, 6), Image.BILINEAR)) for im in x_train[keep][:64]]) / 255.0
y = (y_train[keep][:64] == 1).astype(float)
print(small.shape, "first targets:", y[:8])
print(small[0].round(2))

(64, 6, 6) first targets: [0. 1. 1. 1. 1. 0. 1. 1.]
[[0.   0.   0.01 0.13 0.05 0.  ]
 [0.   0.01 0.27 0.69 0.37 0.01]
 [0.   0.18 0.47 0.24 0.47 0.06]
 [0.   0.38 0.14 0.13 0.4  0.04]
 [0.01 0.4  0.49 0.36 0.08 0.  ]
 [0.   0.08 0.18 0.03 0.   0.  ]]


## Forward propagation, one image

In [3]:
rng = np.random.default_rng(0)
W1 = rng.normal(0, 0.5, (3, 3)); b1 = 0.1             # the filter and its bias
W2 = rng.normal(0, 0.5, (1, 4)); b2 = 0.0             # the output node's 4 weights and its bias

def conv2d(X, K):
    n = X.shape[0] - K.shape[0] + 1
    return np.array([[(X[i:i + K.shape[0], j:j + K.shape[1]] * K).sum() for j in range(n)] for i in range(n)])

def maxpool(A):
    return A.reshape(2, 2, 2, 2).max(axis=(1, 3))     # 4 x 4 -> 2 x 2, windows 2 x 2, stride 2

def forward(X):
    Z1 = conv2d(X, W1) + b1
    A1 = np.maximum(Z1, 0)
    P1 = maxpool(A1)
    F = P1.reshape(4, 1)                               # flatten to a column
    Z2 = W2 @ F + b2
    A2 = 1 / (1 + np.exp(-Z2))
    return Z1, A1, P1, F, Z2, A2

X, t = small[0], y[0]
Z1, A1, P1, F, Z2, A2 = forward(X)
for name, v in zip(("X", "Z1", "A1", "P1", "F", "Z2", "A2"), (X, Z1, A1, P1, F, Z2, A2)):
    print(f"{name:3s} shape {v.shape}")
loss = -(t * np.log(A2) + (1 - t) * np.log(1 - A2))
print("prediction", A2.item(), " target", t, " loss", loss.item())
print("trainable parameters:", W1.size + 1 + W2.size + 1)

X   shape (6, 6)
Z1  shape (4, 4)
A1  shape (4, 4)
P1  shape (2, 2)
F   shape (4, 1)
Z2  shape (1, 1)
A2  shape (1, 1)
prediction 0.21035803598460467  target 0.0  loss 0.23617564635890523
trainable parameters: 15


## The last layer's gradients by hand: dL/dW2 = (a2 - y) F^T, dL/db2 = a2 - y

In [4]:
dW2 = (A2 - t) @ F.T
print("F^T =", F.T.round(4), " a2 - y =", (A2 - t).round(4))
db2 = (A2 - t).item()
print("dL/dW2 =", dW2.round(5), " shape", dW2.shape)
print("dL/db2 =", round(db2, 5))

F^T = [[0.4947 0.4006 0.5311 0.7707]]  a2 - y = [[0.2104]]
dL/dW2 = [[0.10406 0.08428 0.11172 0.16211]]  shape (1, 4)
dL/db2 = 0.21036


## The same gradients from TensorFlow

In [5]:
def tf_forward(X, W1, b1, W2, b2):
    Z1 = tf.nn.conv2d(X[None, :, :, None], W1[:, :, None, None], 1, "VALID")[0, :, :, 0] + b1
    P1 = tf.nn.max_pool2d(tf.nn.relu(Z1)[None, :, :, None], 2, 2, "VALID")[0, :, :, 0]
    return tf.sigmoid(W2 @ tf.reshape(P1, (4, 1)) + b2)

v = [tf.Variable(np.array(a, float)) for a in (W1, b1, W2, b2)]
with tf.GradientTape() as tape:
    a2 = tf_forward(tf.constant(X), *v)
    L = -(t * tf.math.log(a2) + (1 - t) * tf.math.log(1 - a2))
g = tape.gradient(L, v)
print("TensorFlow dL/dW2 =", g[2].numpy().round(5))
print("TensorFlow dL/db2 =", g[3].numpy().round(5).ravel())
print("largest difference:", max(np.abs(g[2].numpy() - dW2).max(), abs(g[3].numpy().item() - db2)))

TensorFlow dL/dW2 = [[0.10406 0.08428 0.11172 0.16211]]
TensorFlow dL/db2 = [0.21036]
largest difference: 0.0


## A batch of m images: F is 4 x m, the gradient stays 1 x 4
With the loss averaged over the batch, a 1/m appears: dL/dW2 = (1/m) (A2 - Y) F^T.

In [6]:
m = 32
Fs = np.hstack([forward(small[i])[3] for i in range(m)])           # 4 x m
A2s = 1 / (1 + np.exp(-(W2 @ Fs + b2)))                               # 1 x m
Y = y[:m].reshape(1, m)
dW2_batch = (A2s - Y) @ Fs.T / m
print("F", Fs.shape, " A2", A2s.shape, " dL/dW2", dW2_batch.shape)
with tf.GradientTape() as tape:
    preds = tf.concat([tf_forward(tf.constant(small[i]), *v) for i in range(m)], axis=1)
    L = tf.reduce_mean(-(Y * tf.math.log(preds) + (1 - Y) * tf.math.log(1 - preds)))
g = tape.gradient(L, v)
print("largest difference from TensorFlow:", np.abs(g[2].numpy() - dW2_batch).max(),
      abs(g[3].numpy().item() - (A2s - Y).mean()))
pd.DataFrame({"name": ["dW2_1", "dW2_2", "dW2_3", "dW2_4", "db2"], "value": list(dW2.ravel()) + [db2]}).to_csv(
    DATA / "last_layer_grads.csv", index=False)

F (4, 32)  A2 (1, 32)  dL/dW2 (1, 4)


largest difference from TensorFlow: 2.0816681711721685e-17 5.551115123125783e-17


In [7]:
# Worked numbers quoted in the Note, step by step (first image, starting weights)
np.set_printoptions(precision=4, suppress=True)
print("top-left 3 x 3 window of X:\n", X[:3, :3].round(4))
print("W1:\n", W1.round(4), "  b1 =", b1)
print("the 9 products:", (X[:3, :3] * W1).round(4).ravel(), " sum", round((X[:3, :3] * W1).sum(), 4),
      " plus b1 ->", round((X[:3, :3] * W1).sum() + b1, 4))
print("Z1:\n", Z1.round(4))
print("all Z1 > 0:", bool((Z1 > 0).all()), "  P1:\n", P1.round(4))
print("W2:", W2.round(4), "  b2 =", b2)
print("the 4 products of Z2:", (W2.ravel() * F.ravel()).round(4), " Z2 =", Z2.round(4).item())
a2 = A2.item()
print("a2 =", round(a2, 4), " loss =", round(loss.item(), 4))
print("dL/da2 = -y/a2 + (1-y)/(1-a2) =", round(-t / a2 + (1 - t) / (1 - a2), 4))
print("da2/dZ2 = a2(1-a2) =", round(a2 * (1 - a2), 4))
# a batch of two images: the first (a 0) and the second (a 1)
F2 = np.hstack([forward(small[i])[3] for i in range(2)])           # 4 x 2
A22 = 1 / (1 + np.exp(-(W2 @ F2 + b2)))                              # 1 x 2
Y2 = y[:2].reshape(1, 2)
print("second image: target", y[1], " F", F2[:, 1].round(4), " a2", A22[0, 1].round(4), " a2 - y", (A22 - Y2)[0, 1].round(4))
print("per-image dL/dW2:\n", ((A22 - Y2) * F2).T.round(4))
print("batch of 2: dL/dW2 =", ((A22 - Y2) @ F2.T / 2).round(4), " dL/db2 =", (A22 - Y2).mean().round(4))

top-left 3 x 3 window of X:
 [[0.     0.     0.0078]
 [0.     0.0078 0.2706]
 [0.     0.1765 0.4667]]
W1:
 [[ 0.0629 -0.0661  0.3202]
 [ 0.0525 -0.2678  0.1808]
 [ 0.652   0.4735 -0.3519]]   b1 = 0.1
the 9 products: [ 0.     -0.      0.0025  0.     -0.0021  0.0489  0.      0.0836 -0.1642]  sum -0.0313  plus b1 -> 0.0687
Z1:
 [[0.0687 0.4468 0.256  0.4006]
 [0.3558 0.4947 0.2482 0.2801]
 [0.1859 0.5311 0.7707 0.2888]
 [0.0771 0.2393 0.2995 0.1123]]
all Z1 > 0: True   P1:
 [[0.4947 0.4006]
 [0.5311 0.7707]]
W2: [[-0.6327 -0.3116  0.0207 -1.1625]]   b2 = 0.0
the 4 products of Z2: [-0.313  -0.1249  0.011  -0.8959]  Z2 = -1.3228
a2 = 0.2104  loss = 0.2362
dL/da2 = -y/a2 + (1-y)/(1-a2) = 1.2664
da2/dZ2 = a2(1-a2) = 0.1661
second image: target 1.0  F [0.3766 0.5051 0.5133 0.4346]  a2 0.291  a2 - y -0.709
per-image dL/dW2:
 [[ 0.1041  0.0843  0.1117  0.1621]
 [-0.267  -0.3581 -0.3639 -0.3081]]
batch of 2: dL/dW2 = [[-0.0815 -0.1369 -0.1261 -0.073 ]]  dL/db2 = -0.2493


## Save the worked image and the starting weights (used by the figure scripts)

In [8]:
np.savez(DATA / "digit_example.npz", X=X, t=t, W1=W1, b1=b1, W2=W2, b2=b2)
print("saved: X", X.shape, " target", t)

saved: X (6, 6)  target 0.0
